# Arkavidia 10.0 - Datavidia Competition
## Team: Jakwan Bagung
### Air Quality Prediction Model

This notebook implements an ensemble prediction model for forecasting air quality (ISPU) categories in Jakarta based on:
- Air quality measurements (PM10, PM2.5, O3, SO2, CO, NO2)
- Weather data (temperature, precipitation, wind, humidity, cloud cover)
- River quality parameters (BOD, heavy metals, pH, TDS, TSS)
- Vegetation index (NDVI)
- Population density by area
- National holiday calendar

## 1. Setup and Imports

In [11]:
import sys
import os

# Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Machine Learning
from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Display Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ All imports successful!")

✓ All imports successful!


## 2. Data Loading

Load all preprocessed datasets:
- **ISPU**: Air quality index data (2010-2024)
- **Weather**: Daily weather observations (2010-2025)
- **River Quality**: Water quality measurements (2015-2025)
- **NDVI**: Vegetation index data
- **Population**: Population data by station
- **Holidays**: National holiday calendar

In [12]:
# Define file paths
file_paths = {
    'ispu': '../ISPU_2010-2024.csv',
    'weather': '../data/cleaned/weather_2010-2025.csv',
    'river': '../data/cleaned/river_quality_2015-2025_daily.csv',
    'holidays': '../data/libur-nasional/libur_nasional.csv',
    'ndvi': '../data/NDVI (vegetation index)/ndvi.csv',
    'population': 'populasi_2010_2025.csv'
}

# Load all datasets
print("Loading datasets...")
print("=" * 60)

df_ispu = pd.read_csv(file_paths['ispu'])
df_ispu['tanggal'] = pd.to_datetime(df_ispu['tanggal'])
print(f"✓ ISPU: {df_ispu.shape[0]:,} records, {df_ispu.shape[1]} columns")

df_weather = pd.read_csv(file_paths['weather'])
df_weather['tanggal'] = pd.to_datetime(df_weather['tanggal'])
if 'stasiun' in df_weather.columns:
    df_weather['stasiun'] = df_weather['stasiun'].str.upper()
print(f"✓ Weather: {df_weather.shape[0]:,} records, {df_weather.shape[1]} columns")

df_river = pd.read_csv(file_paths['river'])
df_river['tanggal'] = pd.to_datetime(df_river['tanggal'])
if 'stasiun_id' in df_river.columns:
    df_river = df_river.rename(columns={'stasiun_id': 'stasiun'})
print(f"✓ River: {df_river.shape[0]:,} records, {df_river.shape[1]} columns")

df_holidays = pd.read_csv(file_paths['holidays'])
df_holidays['tanggal'] = pd.to_datetime(df_holidays['tanggal'])
print(f"✓ Holidays: {df_holidays.shape[0]:,} records")

df_ndvi = pd.read_csv(file_paths['ndvi'])
df_ndvi['tanggal'] = pd.to_datetime(df_ndvi['tanggal'])
if 'stasiun_id' in df_ndvi.columns:
    df_ndvi = df_ndvi.rename(columns={'stasiun_id': 'stasiun'})
df_ndvi['stasiun'] = df_ndvi['stasiun'].str.upper()
print(f"✓ NDVI: {df_ndvi.shape[0]:,} records")

df_population = pd.read_csv(file_paths['population'])
df_population['tanggal'] = pd.to_datetime(df_population['tanggal'])
df_population['stasiun'] = df_population['stasiun'].str.upper()
print(f"✓ Population: {df_population.shape[0]:,} records")

print("=" * 60)
print("✓ All datasets loaded successfully!")

Loading datasets...
✓ ISPU: 14,725 records, 11 columns
✓ Weather: 28,610 records, 27 columns
✓ River: 19,725 records, 47 columns
✓ Holidays: 5,844 records
✓ NDVI: 1,810 records


FileNotFoundError: [Errno 2] No such file or directory: 'populasi_2010_2025.csv'

## 3. Data Merging and Integration

Merge all datasets on `[tanggal, stasiun]` keys to create a unified training dataset.

In [ ]:
print("Merging datasets...")

# Normalize date columns
df_ispu['tanggal'] = pd.to_datetime(df_ispu['tanggal']).dt.normalize()
df_weather['tanggal'] = pd.to_datetime(df_weather['tanggal']).dt.normalize()
df_river['tanggal'] = pd.to_datetime(df_river['tanggal']).dt.normalize()
df_ndvi['tanggal'] = pd.to_datetime(df_ndvi['tanggal']).dt.normalize()
df_population['tanggal'] = pd.to_datetime(df_population['tanggal']).dt.normalize()
df_holidays['tanggal'] = pd.to_datetime(df_holidays['tanggal']).dt.normalize()

# Step 1: Merge ISPU + Weather
merged = pd.merge(df_ispu, df_weather, on=['tanggal', 'stasiun'], how='left')
print(f"After ISPU + Weather merge: {merged.shape}")

# Step 2: Add River Quality
# Force river columns to numeric and aggregate duplicates
river_cols = ['biological_oxygen_demand', 'cadmium', 'chemical_oxygen_demand', 'chromium_vi',
              'copper', 'fecal_coliform', 'lead', 'mbas_detergent', 'mercury',
              'oil_and_grease', 'ph', 'total_coliform', 'total_dissolved_solids',
              'total_suspended_solids', 'zinc']
for col in river_cols:
    if col in df_river.columns:
        df_river[col] = pd.to_numeric(df_river[col], errors='coerce')
        
# Aggregate river data to remove duplicates
df_river_clean = df_river.groupby(['tanggal', 'stasiun'], as_index=False)[[c for c in river_cols if c in df_river.columns]].mean()

merged = pd.merge(merged, df_river_clean, on=['tanggal', 'stasiun'], how='left')
print(f"After adding River Quality: {merged.shape}")

# Step 3: Add National Holidays
merged = pd.merge(merged, df_holidays, on=['tanggal'], how='left')
# Fill missing holiday indicators
if 'is_holiday_nasional' in merged.columns:
    merged['is_holiday_nasional'] = merged['is_holiday_nasional'].fillna(0).astype(int)
if 'nama_libur' in merged.columns:
    merged['nama_libur'] = merged['nama_libur'].fillna('')
if 'is_weekend' not in merged.columns:
    merged['is_weekend'] = (merged['tanggal'].dt.dayofweek >= 5).astype(int)
else:
    merged['is_weekend'] = merged['is_weekend'].fillna((merged['tanggal'].dt.dayofweek >= 5).astype(int)).astype(int)
print(f"After adding Holidays: {merged.shape}")

# Step 4: Add NDVI with duplicate removal
# Clean dates and remove duplicates
df_ndvi['tanggal'] = pd.to_datetime(df_ndvi['tanggal']).dt.normalize()
df_ndvi_clean = df_ndvi.groupby(['tanggal', 'stasiun'], as_index=False)['ndvi'].mean()

merged = pd.merge(merged, df_ndvi_clean, on=['tanggal', 'stasiun'], how='left')
print(f"After adding NDVI: {merged.shape}")

# Step 5: Add Population
merged = pd.merge(merged, df_population, on=['tanggal', 'stasiun'], how='left')
print(f"After adding Population: {merged.shape}")

# Add temporal features
merged['doy'] = merged['tanggal'].dt.dayofyear
merged['month'] = merged['tanggal'].dt.month
merged['year'] = merged['tanggal'].dt.year

# Clean up columns
if 'ID_ispu' in merged.columns:
    merged = merged.rename(columns={'ID_ispu': 'ID'})
if 'day_name' in merged.columns:
    merged = merged.drop(columns=['day_name'])

# Filter out invalid categories
if 'kategori' in merged.columns:
    # Drop "TIDAK ADA DATA" rows
    merged = merged[merged['kategori'] != 'TIDAK ADA DATA']
    # Group extreme categories into "TIDAK SEHAT"
    grouping_map = {
        'SANGAT TIDAK SEHAT': 'TIDAK SEHAT',
        'BERBAHAYA': 'TIDAK SEHAT'
    }
    merged['kategori'] = merged['kategori'].replace(grouping_map)

print(f"\nFinal merged dataset: {merged.shape}")
print(f"Date range: {merged['tanggal'].min()} to {merged['tanggal'].max()}")
print(f"Stations: {sorted(merged['stasiun'].unique())}")
print(f"\nCategory distribution:")
print(merged['kategori'].value_counts())

# Store as training data
df_train = merged.copy()

## 4. Missing Data Handling

Interpolate missing values using time-based interpolation within each station.

In [ ]:
print("Handling missing values...")

# Sort by station and date for proper time-series interpolation
df_train = df_train.sort_values(by=['stasiun', 'tanggal'])

# Identify columns with missing data
cols_to_interpolate = [
    'pm_sepuluh', 'sulfur_dioksida', 'karbon_monoksida', 'ozon', 
    'nitrogen_dioksida', 'chromium_vi', 'ndvi'
]

def fill_gaps_by_station(group):
    """Interpolate missing values within each station using time-based method"""
    group = group.set_index('tanggal')
    # Time interpolation respects actual time gaps
    for col in cols_to_interpolate:
        if col in group.columns:
            group[col] = group[col].interpolate(method='time', limit_direction='both')
    return group.reset_index()

# Apply interpolation per station
df_train = df_train.groupby('stasiun', group_keys=False).apply(fill_gaps_by_station)

# Drop any remaining rows with missing target or critical features
df_train = df_train.dropna(subset=['kategori'])

print(f"Final training data after interpolation: {df_train.shape}")
print(f"\nCategory distribution:")
print(df_train['kategori'].value_counts())

## 5. Test Data Preparation and Extension

Load test data and forecast missing features using regression models trained on historical patterns.

In [ ]:
# Load test data
print("Loading test data...")
df_test = pd.read_csv('../data/sample_submission.csv')
df_test['tanggal'] = pd.to_datetime(df_test['id'].str.split('_').str[0])
df_test['stasiun'] = df_test['id'].str.split('_').str[1].str.upper()
df_test['doy'] = df_test['tanggal'].dt.dayofyear
df_test['month'] = df_test['tanggal'].dt.month
df_test['year'] = df_test['tanggal'].dt.year
df_test['is_weekend'] = (df_test['tanggal'].dt.dayofweek >= 5).astype(int)

print(f"Test data: {df_test.shape}")
print(f"Date range: {df_test['tanggal'].min()} to {df_test['tanggal'].max()}")
print(f"Stations: {sorted(df_test['stasiun'].unique())}")

In [ ]:
print("Forecasting weather features using seasonal patterns...")

# Identify weather columns from training data
weather_cols = [col for col in df_train.columns if any(x in col for x in 
    ['temperature', 'precipitation', 'wind', 'humidity', 'cloud', 'radiation', 'pressure'])]

# Create cloud_cover_mean if it doesn't exist (matched to Untitled7)
if 'cloud_cover_mean (%)' not in df_train.columns:
    if 'cloud_cover_max (%)' in df_train.columns and 'cloud_cover_min (%)' in df_train.columns:
        print("   Computing cloud_cover_mean from max/min...")
        df_train['cloud_cover_mean (%)'] = (df_train['cloud_cover_max (%)'] + df_train['cloud_cover_min (%)']) / 2
        if 'cloud_cover_mean (%)' not in weather_cols:
            weather_cols.append('cloud_cover_mean (%)')

# Create seasonal weather averages (by station and day of year)
seasonal_weather = df_train.groupby(['stasiun', 'doy'])[weather_cols].mean().reset_index()

# Apply seasonal patterns to test set
for col in weather_cols:
    if col in seasonal_weather.columns:
        # Create mapping: (Station, DOY) -> Average Value
        mapping = seasonal_weather.set_index(['stasiun', 'doy'])[col]
        df_test[col] = df_test.set_index(['stasiun', 'doy']).index.map(mapping).values
        # Fill any remaining NaNs with station-level average
        if df_test[col].isnull().any():
            station_avg = df_train.groupby('stasiun')[col].mean()
            df_test[col] = df_test[col].fillna(df_test['stasiun'].map(station_avg))

# Compute cloud_cover_mean for test set if needed
if 'cloud_cover_mean (%)' not in df_test.columns:
    if 'cloud_cover_max (%)' in df_test.columns and 'cloud_cover_min (%)' in df_test.columns:
        df_test['cloud_cover_mean (%)'] = (df_test['cloud_cover_max (%)'] + df_test['cloud_cover_min (%)']) / 2

print(f"✓ {len(weather_cols)} weather features forecasted")

In [ ]:
print("Forecasting pollutants, river quality, and NDVI using XGBoost regressors...")

# Apply one-hot encoding to stations for both train and test
df_train_ohe = pd.get_dummies(df_train, columns=['stasiun'], prefix='stasiun')
df_test_ohe = pd.get_dummies(df_test, columns=['stasiun'], prefix='stasiun')

# Align OHE columns (ensure test has same station columns as train)
for col in [c for c in df_train_ohe.columns if 'stasiun_' in c]:
    if col not in df_test_ohe.columns:
        df_test_ohe[col] = 0
df_test = df_test_ohe.copy()

# Define targets to forecast (pollutants, river, NDVI)
numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
all_numeric_cols = df_train.select_dtypes(include=numerics).columns

# Exclude weather (already forecasted), time features, and PM2.5 (calculated later)
excludes = ['doy', 'month', 'year', 'is_weekend', 'is_holiday_nasional', 'kategori', 'pm_duakomalima'] + weather_cols
targets = [c for c in all_numeric_cols if c not in excludes and 'stasiun_' not in c and 'ID' not in c]

# Features used to predict targets (weather + time + station OHE)
predictor_features = weather_cols + ['doy', 'is_weekend'] + [c for c in df_test.columns if 'stasiun_' in c]

forecast_count = 0
for target in targets:
    # Train regressor on clean training data
    train_subset = df_train_ohe.dropna(subset=[target])
    if len(train_subset) == 0:
        continue
    
    model = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, n_jobs=-1, random_state=42)
    model.fit(train_subset[predictor_features], train_subset[target])
    
    # Predict for test
    df_test[target] = model.predict(df_test[predictor_features])
    forecast_count += 1

print(f"✓ {forecast_count} features forecasted using regression models")

In [ ]:
print("Calculating PM2.5 using physics-based ratio...")

# Calculate historical ratio: PM2.5 / PM10
if 'pm_duakomalima' in df_train.columns and 'pm_sepuluh' in df_train.columns:
    valid_data = df_train.dropna(subset=['pm_duakomalima', 'pm_sepuluh'])
    ratio = (valid_data['pm_duakomalima'] / valid_data['pm_sepuluh'].replace(0, 1)).mean()
    print(f"   Historical PM2.5/PM10 ratio: {ratio:.4f}")
    
    # Apply ratio to forecasted PM10
    df_test['pm_duakomalima'] = df_test['pm_sepuluh'] * ratio
    print("   ✓ PM2.5 calculated successfully")
else:
    df_test['pm_duakomalima'] = 0
    print("   WARNING: PM2.5 columns missing, setting to 0")

## 5.5 Model Evaluation and Feature Analysis

Before training on the full dataset, let's validate our approach using a train/validation split.

In [ ]:
print("Preparing validation data...")

# Encode target
le_eval = LabelEncoder()
df_train_clean_eval = df_train_ohe.dropna(subset=['kategori']).copy()
y = le_eval.fit_transform(df_train_clean_eval['kategori'])

# Define features (same as we'll use for final model)
exclude_cols = ['id', 'ID', 'tanggal', 'kategori', 'category_encoded', 'year', 'month', 'nama_libur']
features_eval = [col for col in df_test.columns
                 if col in df_train_clean_eval.columns and col not in exclude_cols]

print(f"Evaluating with {len(features_eval)} features")
print(f"Target classes: {le_eval.classes_}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

In [ ]:
print("Running validation split (80% train / 20% validation)...")

# Split data
X_train_eval, X_val_eval, y_train_eval, y_val_eval = train_test_split(
    df_train_clean_eval[features_eval], y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# Train model for evaluation (matched to Untitled7)
clf_eval = XGBClassifier(
    n_estimators=300, 
    learning_rate=0.05, 
    max_depth=6,
    subsample=0.8, 
    n_jobs=-1, 
    random_state=42
)

print("Training evaluation model...")
clf_eval.fit(X_train_eval, y_train_eval)

# Predict on validation set
y_pred_eval = clf_eval.predict(X_val_eval)

print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)
print("\nClassification Report:")
print(classification_report(y_val_eval, y_pred_eval, target_names=le_eval.classes_))

# Calculate overall accuracy
from sklearn.metrics import accuracy_score
val_accuracy = accuracy_score(y_val_eval, y_pred_eval)
print(f"\nValidation Accuracy: {val_accuracy:.2%}")

In [ ]:
print("Generating confusion matrix...")

# Confusion Matrix
cm = confusion_matrix(y_val_eval, y_pred_eval)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_eval.classes_, yticklabels=le_eval.classes_,
            cbar_kws={'label': 'Count'})
plt.title("Confusion Matrix (Validation Set)", fontsize=14, fontweight='bold')
plt.xlabel("Predicted Category", fontsize=12)
plt.ylabel("Actual Category", fontsize=12)
plt.tight_layout()
plt.show()

print("\nConfusion Matrix Interpretation:")
print(f"- Correct BAIK predictions: {cm[0,0]}/{cm[0].sum()} ({cm[0,0]/cm[0].sum():.1%})")
print(f"- Correct SEDANG predictions: {cm[1,1]}/{cm[1].sum()} ({cm[1,1]/cm[1].sum():.1%})")
print(f"- Correct TIDAK SEHAT predictions: {cm[2,2]}/{cm[2].sum()} ({cm[2,2]/cm[2].sum():.1%})")

In [ ]:
print("Analyzing feature correlations with target variable...")

# Add encoded target to training data for correlation analysis
df_train_clean_eval['category_encoded'] = y

# Calculate correlations
corr_data = df_train_clean_eval[features_eval + ['category_encoded']].corr()['category_encoded'].sort_values(ascending=False)

print("\nTarget variable encoding:")
print(dict(zip(le_eval.classes_, le_eval.transform(le_eval.classes_))))
print("\nTop 20 Positive Correlations (Higher values → Worse air quality):")
print(corr_data.drop('category_encoded').head(20))

print("\nTop 10 Negative Correlations (Higher values → Better air quality):")
print(corr_data.drop('category_encoded').tail(10))

# Visualize top correlations
plt.figure(figsize=(12, 8))
top_corr = corr_data.drop('category_encoded').head(15)  # Top 15 positive
bottom_corr = corr_data.drop('category_encoded').tail(5)  # Top 5 negative
plot_corr = pd.concat([top_corr, bottom_corr])

sns.barplot(x=plot_corr.values, y=plot_corr.index, palette='coolwarm')
plt.title("Top Feature Correlations with Air Quality Category", fontsize=14, fontweight='bold')
plt.xlabel("Correlation Coefficient", fontsize=12)
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("Analyzing feature importance from XGBoost model...")

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': features_eval,
    'importance': clf_eval.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(feature_importance.head(20).to_string(index=False))

# Visualize top 20 features
plt.figure(figsize=(12, 8))
top_20_features = feature_importance.head(20)
sns.barplot(data=top_20_features, x='importance', y='feature', palette='viridis')
plt.title("Top 20 Most Important Features (XGBoost)", fontsize=14, fontweight='bold')
plt.xlabel("Feature Importance", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.tight_layout()
plt.show()

print(f"\n✓ Evaluation complete! Validation accuracy: {val_accuracy:.2%}")

## 6. Model Training and Prediction

Train an optimized XGBoost classifier and blend predictions with historical patterns for robust forecasting.

In [ ]:
# ==============================================================================
# 1. DEFINING THE "PROVEN" FEATURES
# ==============================================================================
print("1. Configuring Optimized Feature Set...")

# Proven Features from Correlation Analysis
optimized_features = [
    'pm_sepuluh', 'pm_duakomalima', 'ozon',
    'nitrogen_dioksida', 'karbon_monoksida', 'sulfur_dioksida',
    'temperature_2m_max (°C)', 'shortwave_radiation_sum (MJ/m²)', 'temperature_2m_mean (°C)',
    'wind_gusts_10m_mean (km/h)', 'wind_speed_10m_min (km/h)',
    'relative_humidity_2m_min (%)', 'cloud_cover_min (%)', 'cloud_cover_mean (%)',
    'precipitation_hours (h)', 'precipitation_sum (mm)',
    'ndvi', 'lead', 'mercury', 'doy',
    *[c for c in df_test.columns if 'stasiun_' in c]
]

# ==============================================================================
# 2. TRAIN MODEL
# ==============================================================================
print(f"2. Training on {len(optimized_features)} features...")

le = LabelEncoder()
df_train_clean = df_train_ohe.dropna(subset=['kategori']).copy()
y_train = le.fit_transform(df_train_clean['kategori'])

final_features = [f for f in optimized_features if f in df_train_clean.columns and f in df_test.columns]

clf = XGBClassifier(n_estimators=350, learning_rate=0.04, max_depth=6, subsample=0.8, n_jobs=-1, random_state=42)
clf.fit(df_train_clean[final_features], y_train)

print(f"✓ Model trained with {len(final_features)} features")

In [ ]:
# ==============================================================================
# 3. BLEND (Golden Ratio)
# ==============================================================================
print("3. Blending with History...")

xgb_probs = clf.predict_proba(df_test[final_features])

# History Probs
if 'stasiun' not in df_train_clean.columns:
    stasiun_cols = [c for c in df_train_clean.columns if 'stasiun_' in c]
    df_train_clean['stasiun'] = df_train_clean[stasiun_cols].idxmax(axis=1).str.replace('stasiun_', '')
if 'stasiun' not in df_test.columns:
    df_test['stasiun'] = df_test['id'].str.split('_').str[1]

df_train_clean['month'] = df_train_clean['tanggal'].dt.month
history_probs = df_train_clean.groupby(['stasiun', 'month'])['kategori'].value_counts(normalize=True).unstack(fill_value=0)

df_test['month'] = df_test['tanggal'].dt.month
test_history = pd.merge(df_test[['stasiun', 'month']], history_probs, on=['stasiun', 'month'], how='left')
test_history = test_history[['BAIK', 'SEDANG', 'TIDAK SEHAT']].fillna(0.33)
hist_probs_array = test_history.values

# MIX: 80% History / 20% Smart Model
alpha = 0.2
final_probs = (alpha * xgb_probs) + ((1 - alpha) * hist_probs_array)

print(f"✓ Blending complete with alpha={alpha} (20% model + 80% history)")

In [ ]:
# ==============================================================================
# 4. SAMPLE & SMOOTH (Fixed Logic)
# ==============================================================================
print("4. Sampling and Smoothing...")

# A. Sample
np.random.seed(42)
raw_predictions = []
classes = le.classes_

for p in final_probs:
    p = p / p.sum()
    pred = np.random.choice(classes, p=p)
    raw_predictions.append(pred)

# Create Temp DF
temp_df = df_test[['id']].copy()
temp_df['category'] = raw_predictions
temp_df['tanggal'] = pd.to_datetime(temp_df['id'].str.split('_').str[0])
temp_df['stasiun'] = temp_df['id'].str.split('_').str[1]
temp_df = temp_df.sort_values(['stasiun', 'tanggal'])

# Map to numbers
cat_map = {'BAIK': 0, 'SEDANG': 1, 'TIDAK SEHAT': 2}
rev_map = {0: 'BAIK', 1: 'SEDANG', 2: 'TIDAK SEHAT'}
temp_df['cat_code'] = temp_df['category'].map(cat_map)

# --- THE FIX: Function accepts the SERIES directly ---
def smooth_series(series):
    vals = series.values.copy() # Work on numpy array
    # Fix "Bad -> Good -> Bad" flicker
    for i in range(1, len(vals)-1):
        if vals[i-1] == 2 and vals[i+1] == 2 and vals[i] != 2:
            vals[i] = 2
    return vals

# Apply transform on the COLUMN 'cat_code'
temp_df['new_code'] = temp_df.groupby('stasiun')['cat_code'].transform(smooth_series)
temp_df['category'] = temp_df['new_code'].map(rev_map)

print("✓ Temporal smoothing complete")

## 7. Generate Submission

Create final submission file with predicted air quality categories.

In [ ]:
# ==============================================================================
# 5. SAVE
# ==============================================================================
submission = temp_df[['id', 'category']].copy()

# Merge with sample submission to ensure correct order
sample_sub = pd.read_csv('../data/sample_submission.csv')
final_submission = pd.merge(sample_sub[['id']], submission, on='id', how='left')
final_submission['category'] = final_submission['category'].fillna('SEDANG')

# Save submission file
output_file = '../submission_jakwan_bagung.csv'
final_submission.to_csv(output_file, index=False)

print("\n" + "="*60)
print(f"✓ SUCCESS! Saved '{output_file}'")
print("="*60)
print(f"\nTotal predictions: {len(final_submission)}")
print("\n--- Final Distribution ---")
print(final_submission['category'].value_counts())
print(f"\nPercentages:")
print((final_submission['category'].value_counts(normalize=True) * 100).round(2))
print("\n--- Station Breakdown (TIDAK SEHAT) ---")
ts_by_station = final_submission[final_submission['category']=='TIDAK SEHAT']['id'].str.split('_').str[1].value_counts()
print(ts_by_station)
print("="*60)

In [ ]:
# Visualization: Distribution by station and category
final_submission['stasiun'] = final_submission['id'].str.split('_').str[1]
final_submission['tanggal'] = pd.to_datetime(final_submission['id'].str.split('_').str[0])

print("\n" + "=" * 60)
print("Station-Level Distribution of TIDAK SEHAT predictions:")
print("=" * 60)
unhealthy_by_station = final_submission[final_submission['category'] == 'TIDAK SEHAT']['stasiun'].value_counts()
print(unhealthy_by_station)

# Plot category distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Overall distribution
final_submission['category'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'yellow', 'red'])
axes[0].set_title('Overall Category Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Distribution by station
station_category = final_submission.groupby(['stasiun', 'category']).size().unstack(fill_value=0)
station_category.plot(kind='bar', stacked=True, ax=axes[1], color=['green', 'yellow', 'red'])
axes[1].set_title('Category Distribution by Station', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Station')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Category')

plt.tight_layout()
plt.show()

print("\n✓ Analysis complete!")

## Model Methodology Summary

### Data Sources
- **ISPU**: Air quality measurements (PM10, PM2.5, O3, SO2, CO, NO2) from 5 Jakarta stations (2010-2024)
- **Weather**: Daily meteorological data (temperature, precipitation, wind, humidity, cloud cover) (2010-2025)
- **River Quality**: Water quality parameters (BOD, heavy metals, pH, TDS, TSS) (2015-2025)
- **NDVI**: Vegetation index from satellite imagery
- **Population**: Daily population estimates by station area
- **Holidays**: National holiday calendar for temporal context

### Preprocessing & Feature Engineering
1. **Data Integration**: 
   - Merged all datasets on `[tanggal, stasiun]` keys using left joins
   - Removed duplicate entries by aggregating on date-station pairs
   - Filtered out invalid categories ("TIDAK ADA DATA") and grouped extreme categories into "TIDAK SEHAT"
   
2. **Missing Value Handling**: 
   - Time-based interpolation within each station for continuity
   - Cleaned and aggregated river quality data with numeric conversion
   
3. **Test Data Extension**: 
   - Weather forecasted using seasonal patterns (DOY-based averages)
   - Pollutants/river/NDVI forecasted using XGBoost regressors
   - PM2.5 calculated from PM10 using historical ratio
   
4. **Feature Engineering**:
   - One-hot encoding for station locations
   - Temporal features (day of year)
   - Cloud cover mean calculated from max/min values
   - **20 optimized features** selected based on correlation analysis:
     * Core pollutants: PM10, PM2.5, O3, NO2, CO, SO2
     * Environmental: Lead, Mercury, NDVI
     * Weather: Temperature max/mean, Radiation, Wind speed/gusts, Humidity min, Cloud cover mean/min, Precipitation sum/hours
     * Temporal: Day of year
     * Location: Station one-hot encoding

### Model Architecture
- **Base Model**: XGBoost Classifier (350 trees, lr=0.04, depth=6, subsample=0.8)
- **Blending Strategy**: 20% model predictions + 80% historical baseline
  - Historical baseline: Monthly category frequencies by station
  - Conservative approach to leverage stable patterns
- **Post-processing**: Temporal smoothing to eliminate day-to-day flicker
  - Fixed "TIDAK SEHAT → BAIK → TIDAK SEHAT" transitions
  - Maintains realistic temporal continuity

### Key Design Choices
- Single optimized model (not ensemble) for simplicity and interpretability
- Golden ratio blending (80/20) favors historical stability over model volatility
- Feature set refined through correlation analysis to include only proven predictors
- Robust to distribution shift through strong historical component